# Generative AI: Transformer-Based Text Generation with GPT-2

**Author:** Roberto Jourdain

**Model:** GPT-2 (pretrained) — sourced from [HuggingFace](https://huggingface.co/openai-community/gpt2)

This project demonstrates transformer-based text generation using OpenAI's GPT-2 language model. The model generates coherent text continuations from user-provided prompts, and the notebook explores how different decoding strategies (greedy, temperature sampling, top-k, and top-p/nucleus sampling) affect output quality, diversity, and coherence.

## Model Loading and Inspection

In [1]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# Load pretrained GPT-2 model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

print(f"Model: GPT-2 (124M parameters)")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"Max sequence length: {model.config.n_positions}")
print(f"Number of layers: {model.config.n_layer}")
print(f"Number of attention heads: {model.config.n_head}")
print(f"Embedding dimension: {model.config.n_embd}")

Model: GPT-2 (124M parameters)
Vocabulary size: 50,257
Max sequence length: 1024
Number of layers: 12
Number of attention heads: 12
Embedding dimension: 768


**Inspection notes:** GPT-2 is a decoder-only transformer with 124 million parameters, a vocabulary of 50,257 tokens (byte-pair encoding), 12 attention layers, 12 heads per layer, and an embedding dimension of 768. The model was pretrained on WebText, a dataset of ~8 million web pages. Since we are using a pretrained model, no separate training dataset is required -- the "data source" is the WebText corpus documented by OpenAI. We use the model in evaluation mode (no gradient computation) for inference-only text generation.

### Training Diagnostics

GPT-2 is a pretrained model, so no training loop is executed in this notebook. Instead, we evaluate the model's pretrained quality by computing **perplexity** on sample sentences. Perplexity measures how well the model predicts a sequence -- lower values indicate the model assigns higher probability to real text, confirming the pretrained weights are functioning correctly.

In [3]:
# Compute perplexity on sample sentences to verify pretrained model quality
test_sentences = [
    "The weather today is sunny and warm with clear skies.",
    "Machine learning models require large amounts of training data.",
    "Purple elephants danced quietly on the surface of the moon.",
]

print("Perplexity of sample sentences (lower = model finds it more probable):\n")
for sentence in test_sentences:
    inputs = tokenizer.encode(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = model(inputs, labels=inputs)
        loss = outputs.loss
        perplexity = torch.exp(loss).item()
    print(f"  Perplexity: {perplexity:>8.2f}  |  \"{sentence}\"")

print("\nNote: Coherent, common sentences have low perplexity. Unusual or")
print("nonsensical sentences have high perplexity, confirming the model has")
print("learned meaningful language patterns from its pretraining data.")

Perplexity of sample sentences (lower = model finds it more probable):



`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  Perplexity:    32.48  |  "The weather today is sunny and warm with clear skies."
  Perplexity:    38.94  |  "Machine learning models require large amounts of training data."
  Perplexity:    96.70  |  "Purple elephants danced quietly on the surface of the moon."

Note: Coherent, common sentences have low perplexity. Unusual or
nonsensical sentences have high perplexity, confirming the model has
learned meaningful language patterns from its pretraining data.


## Text Generation Implementation

The core generation function wraps the HuggingFace `model.generate()` API. We explore four decoding strategies:

1. **Greedy decoding** -- always picks the highest-probability next token. Deterministic but repetitive.
2. **Temperature sampling** -- scales the probability distribution before sampling. Higher temperature = more randomness.
3. **Top-k sampling** -- restricts sampling to the top k most likely tokens, filtering out low-probability noise.
4. **Top-p (nucleus) sampling** -- dynamically selects the smallest set of tokens whose cumulative probability exceeds p, adapting the candidate pool to each prediction step.

In [4]:
def generate_text(prompt, max_length=150, temperature=1.0, top_k=0, top_p=1.0,
                  do_sample=True, num_return_sequences=1):
    """Generate text from a prompt using GPT-2 with configurable decoding parameters.

    Args:
        prompt: Input text to continue from.
        max_length: Maximum total length (prompt + generated tokens).
        temperature: Sampling temperature (higher = more random).
        top_k: If > 0, only sample from the top k tokens.
        top_p: If < 1.0, use nucleus sampling with this cumulative probability threshold.
        do_sample: If False, use greedy decoding.
        num_return_sequences: Number of independent sequences to generate.

    Returns:
        List of generated text strings.
    """
    inputs = tokenizer.encode(prompt, return_tensors="pt")
    attention_mask = torch.ones_like(inputs)

    gen_kwargs = {
        "attention_mask": attention_mask,
        "max_length": max_length,
        "num_return_sequences": num_return_sequences,
        "pad_token_id": tokenizer.eos_token_id,
        "do_sample": do_sample,
    }

    # Only pass sampling parameters when sampling is enabled
    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_k"] = top_k
        gen_kwargs["top_p"] = top_p

    with torch.no_grad():
        outputs = model.generate(inputs, **gen_kwargs)

    return [tokenizer.decode(seq, skip_special_tokens=True) for seq in outputs]

## Generate and Evaluate Outputs

We test the model with multiple prompts and decoding strategies to evaluate output quality, coherence, and diversity.

In [5]:
# Example 1: Greedy decoding (deterministic, often repetitive)
prompt = "Artificial intelligence will transform"
print("=== Greedy Decoding ===")
print(f"Prompt: '{prompt}'\n")
result = generate_text(prompt, do_sample=False)
print(result[0])

=== Greedy Decoding ===
Prompt: 'Artificial intelligence will transform'

Artificial intelligence will transform the way we think about the world. It will change how we think about the world. It will change how we think about the world. It will change how we think about the world. It will change how we think about the world. It will change how we think about the world. It will change how we think about the world. It will change how we think about the world. It will change how we think about the world. It will change how we think about the world. It will change how we think about the world. It will change how we think about the world. It will change how we think about the world. It will change how we think about the world. It will change how we think about


In [6]:
# Example 2: Temperature sampling comparison
prompt = "The future of education depends on"
print("=== Temperature Comparison ===")
print(f"Prompt: '{prompt}'\n")

for temp in [0.3, 0.7, 1.2]:
    print(f"--- Temperature = {temp} ---")
    result = generate_text(prompt, temperature=temp)
    print(result[0])
    print()

=== Temperature Comparison ===
Prompt: 'The future of education depends on'

--- Temperature = 0.3 ---
The future of education depends on the ability of parents to make decisions about their children's future," said Dr. Michael J. Leighton, a professor of education at the University of California, Berkeley. "Parents should be able to make informed decisions about their children's future, but they should also be able to make informed decisions about their children's education."

The study, published in the journal Pediatrics, found that parents who were more concerned about their children's future were more likely to report that they would not be able to support their children's education.

"Parents who are more concerned about their children's future are more likely to report that they would not be able to support their children's education," said Dr. Leighton, who

--- Temperature = 0.7 ---
The future of education depends on it," says Prakash, adding it is now possible to have a high-

In [7]:
# Example 3: Top-k sampling
prompt = "Running a marathon requires"
print("=== Top-k Sampling (k=50) ===")
print(f"Prompt: '{prompt}'\n")
result = generate_text(prompt, top_k=50, temperature=0.8)
print(result[0])

=== Top-k Sampling (k=50) ===
Prompt: 'Running a marathon requires'

Running a marathon requires about two hours of running per hour for healthy weightlifters.

To do it correctly, you need to gain at least a day worth of weight in each day.

You need to increase your daily calorie intake by a few thousand calories. For a person to lose the weight of 200 kilos or more, they need to lose 100 kilograms or more.

So in order to maintain a healthy weight for the duration of a marathon, you just need to have a lot of weight in each day.

What's more, by adding a few extra kilos or pounds for every 500 calories left for the marathon, you can increase your daily caloric intake to about half a kilogram.

You'll


In [8]:
# Example 4: Top-p (nucleus) sampling — multiple sequences for diversity check
prompt = "In the year 2050, technology will"
print("=== Top-p (Nucleus) Sampling (p=0.9) — 3 sequences ===")
print(f"Prompt: '{prompt}'\n")
results = generate_text(prompt, top_p=0.9, top_k=0, temperature=0.8, num_return_sequences=3)
for i, text in enumerate(results):
    print(f"--- Sequence {i+1} ---")
    print(text)
    print()

=== Top-p (Nucleus) Sampling (p=0.9) — 3 sequences ===
Prompt: 'In the year 2050, technology will'

--- Sequence 1 ---
In the year 2050, technology will become more useful and it will become cheaper to build the world's second-largest solar energy storage facility.

However, if you want to stay ahead of the curve, you'll need to invest in some efficiency technologies. In order to get that efficiency, you need to invest in the installation of solar panels.

The key to efficiency is power density.

Power density is a measurement of the average amount of energy that a system generates. It can be a measure of the efficiency of a system, such as the amount of heat the system receives in a given amount of time, or the amount of energy in a given amount of electricity.

Generally, in the US, the average power density

--- Sequence 2 ---
In the year 2050, technology will provide a vast amount of space for unprecedentedly large and complex worlds. In this article I will introduce a new space-ba

### Qualitative Evaluation

**Greedy decoding** produces grammatically correct but often repetitive text. It tends to fall into loops where phrases repeat because the model always picks the single most likely next token.

**Temperature** controls the tradeoff between coherence and creativity. Low temperature (0.3) produces safe, predictable text similar to greedy. Medium temperature (0.7) balances coherence and variety. High temperature (1.2) introduces more surprising word choices but risks incoherence.

**Top-k sampling** (k=50) restricts the candidate pool to the 50 most likely tokens at each step, filtering out low-probability noise while maintaining diversity within a reasonable range.

**Top-p (nucleus) sampling** (p=0.9) adapts the candidate pool dynamically -- it includes just enough tokens to cover 90% of the probability mass. This produces the most natural-sounding text because the pool size adjusts based on how confident the model is at each step.

**Common failure cases:** GPT-2 occasionally generates factually incorrect statements, loses coherence over longer sequences, and may produce repetitive structures. These are well-known limitations of autoregressive language models of this scale.

## Summary

This project demonstrated transformer-based text generation using a pretrained GPT-2 model with 124 million parameters. Four decoding strategies were explored: greedy, temperature sampling, top-k, and top-p (nucleus) sampling. The outputs showed a clear tradeoff between coherence and diversity -- greedy decoding is safe but repetitive, while higher temperatures and nucleus sampling produce more varied and natural-sounding text at the cost of occasional incoherence. A key challenge with autoregressive models like GPT-2 is that they have no mechanism for factual grounding -- the model can generate fluent text that is factually incorrect, which poses risks if outputs are used without human review. The model also struggles with long-range coherence, as it has no explicit memory beyond its 1024-token context window.